# E47: tumour size, crop truncation, calibration and final statistics

**Revision plan, 00 log sections 19–20 (approved 17 Oct 2026).** Answers Reviewer 2 point 5 (independently centred
crops, truncation), Reviewer 4 point 8 (large lesions truncated, size differs by class: stratify by size), Reviewer 4
point 6 (small test set, wide intervals) and Reviewer 2 point 1 / Reviewer 4 point 1 (multiple comparisons; the fusion
gain). It uses saved predictions and crops only — **no model is trained or tuned here.**

- **E4a** — how much of each tumour the 96³ crop keeps (per phase and class), how many tumours are larger than the crop,
  and how well the independently centred phases line up inside the crops (tumour and liver overlap between phases).
- **E4b** — internal results by tumour-size thirds (cut-offs from the 222 development patients) and truncated vs not.
- **E4c** — MCT-LTDiag (all 203, from E8.0) by crop retention and by size thirds.
- **E7b** — final internal statistics: bootstrap CIs, per-class sensitivity/specificity with exact CIs, DeLong for all
  pairs of the main models with Benjamini–Hochberg adjustment, paired bootstrap differences.
- **E7c** — calibration (Brier score, intercept, slope, reliability tables), internal test and MCT-LTDiag.
- **E7d** — optimism of the fusion meta-learner (apparent vs cross-validated vs held-out; Harrell bootstrap).

Main models (decided 17 Oct): phase-dropout CNN, radiomics without shape features, CNN + radiomics and full fusion with
the phase-dropout CNN. The original CNN and its fusions are reported alongside.

**Runtime:** CPU is enough (about 15–20 minutes). **Safety:** inputs are only read; outputs go to a new folder
`liver_project/revision_2026-10/E47_<time>/`.

In [ ]:
# Cell 1 — Setup: paths, input checks, output folder
import os, sys, json, time, hashlib, platform, datetime, warnings, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel'], check=True)
else:
    ROOT = Path(os.environ['E47_TEST_ROOT'])

import numpy as np
import pandas as pd
import nibabel as nib
from scipy.stats import beta, norm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
E6_RUN = REVISION_DIR / 'E6_20261017T114646Z'
SEED, N_BOOT, N_OPTIMISM = 42, 2000, 200
CLASSES = ['HCC', 'ICC', 'CHCC']
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']

# The E8a run (E8.0 = E6 on all 203 MCT-LTDiag patients) is the most recent finished E8a folder
e8a_runs = sorted(p for p in REVISION_DIR.glob('E8a_*') if (p / 'summary.json').exists())
if not e8a_runs:
    raise RuntimeError('STOP — no finished E8a run found. Run E8a first.')
E8A_RUN = e8a_runs[-1]

INPUTS = {
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_test_meta': (E2_DIR / 'E2_fusion_test_meta_features_noshape.csv', 'fbb946f641a050f66b093564141a8e103e1c4e5c651d6abeb51042293ab8cb2f'),
    'e3_original_predictions': (E3_DIR / 'E3A_original_cnn_predictions.csv', '5c02c3ebda984b6283c1137d85c17c171347b24ae1fac450dfc89c5513d0229b'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'e6_crop_geometry': (E6_RUN / 'E6_crop_geometry.csv', '11582b534641813a086da8497076dbfd53fe51a3ef1092b12537d55a1f28805b'),
    'e6_facts': (E6_RUN / 'E6_facts_per_patient.csv', 'e0621ba43fc8309b948edf2895c4b82a6908386bee74f890b9df35e4c6868a69'),
    'e8a_predictions_203': (E8A_RUN / 'E8_0_E6_203_predictions.csv', None),
    'e8a_crop_geometry_2': (E8A_RUN / 'E8_0_restored2_crop_geometry.csv', None),
    'e8a_facts_2': (E8A_RUN / 'E8_0_restored2_facts.csv', None),
    'e8a_summary': (E8A_RUN / 'summary.json', None),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
LARGE_INPUTS = {'processed_tar': (PROJECT_DIR / 'processed.tar', 402616320)}

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
for key, (path, size) in LARGE_INPUTS.items():
    if not path.exists() or (IN_COLAB and path.stat().st_size != size):
        problems.append(f'missing or changed size: {path}')
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))
import sklearn, scipy
VERSIONS = {'python': platform.python_version(), 'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'scipy': scipy.__version__}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REVISION_DIR / f'E47_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=False)
print('Using E8a run:', E8A_RUN.name, '| output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Load crops, labels, folds and the saved predictions of every model (internal out-of-fold and held-out)
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(LARGE_INPUTS['processed_tar'][0]), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E47_TEST_PROCESSED'])
split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
LABEL, SIZE, TUMOR_CROP, LIVER_CROP = {}, {}, {}, {}
for pid in dev_ids + test_ids:
    with np.load(PROCESSED_DIR / f'{pid}.npz', allow_pickle=False) as z:
        LABEL[pid] = int(z['label'])
        SIZE[pid] = float(z['tumor_vol_portal'])
        TUMOR_CROP[pid] = z['tumor_mask'].astype(bool)
        LIVER_CROP[pid] = z['liver_mask'].astype(bool)

W4, W5 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC'], ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
P3 = ['p_HCC', 'p_ICC', 'p_CHCC']
oof_meta = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str})
test_meta = pd.read_csv(INPUTS['e2_test_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids].reset_index()
def e3(key, tag, split_name):
    t = pd.read_csv(INPUTS[key][0], dtype={'patient_id': str})
    return t[(t['model'] == tag) & (t['pattern'] == 'complete') & (t['split'] == split_name)].set_index('patient_id')[P3]
D_OOF, D_TEST = e3('e3_d_predictions', 'D_phase_dropout', 'oof'), e3('e3_d_predictions', 'D_phase_dropout', 'test')
O_OOF, O_TEST = e3('e3_original_predictions', 'original', 'oof'), e3('e3_original_predictions', 'original', 'test')

def prepare(X):
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X

def meta_cv(columns, oof, test):
    """E2 meta-learner: 5 fold models on the out-of-fold table; out-of-fold and averaged held-out predictions."""
    oof_p = np.full((len(oof), 3), np.nan)
    test_p = []
    for f in range(5):
        tr, va = oof['fold'].astype(int).ne(f).to_numpy(), oof['fold'].astype(int).eq(f).to_numpy()
        X = prepare(oof.loc[tr, columns].to_numpy())
        imp = SimpleImputer(strategy='median').fit(X)
        sc = StandardScaler().fit(imp.transform(X))
        m = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED).fit(sc.transform(imp.transform(X)), oof.loc[tr, 'label'].astype(int))
        oof_p[va] = m.predict_proba(sc.transform(imp.transform(prepare(oof.loc[va, columns].to_numpy()))))
        test_p.append(m.predict_proba(sc.transform(imp.transform(prepare(test[columns].to_numpy())))))
    return (pd.DataFrame(oof_p, index=oof['patient_id'], columns=P3), pd.DataFrame(np.mean(test_p, axis=0), index=test['patient_id'], columns=P3))

oof_d, test_d = oof_meta.copy(), test_meta.copy()
oof_d[W4] = D_OOF.loc[oof_d['patient_id']].to_numpy()
test_d[W4] = D_TEST.loc[test_d['patient_id']].to_numpy()
PRED = {}                                      # model -> (out-of-fold 222, held-out 56)
PRED['phase_dropout_CNN'] = (D_OOF.loc[dev_ids], D_TEST.loc[test_ids])
PRED['original_CNN'] = (O_OOF.loc[dev_ids], O_TEST.loc[test_ids])
PRED['radiomics_noshape'] = (oof_meta.set_index('patient_id')[W5].set_axis(P3, axis=1).loc[dev_ids],
                             test_meta.set_index('patient_id')[W5].set_axis(P3, axis=1).loc[test_ids])
for name, (o, t) in {'cnn_plus_radiomics [phase_dropout_CNN]': (W4 + W5, None), 'full_fusion [phase_dropout_CNN]': (W4 + W5 + ['age', 'gender'], None)}.items():
    a, b = meta_cv(o, oof_d, test_d)
    PRED[name] = (a.loc[dev_ids], b.loc[test_ids])
for name, cols in {'cnn_plus_radiomics [original_CNN]': W4 + W5, 'full_fusion [original_CNN]': W4 + W5 + ['age', 'gender']}.items():
    a, b = meta_cv(cols, oof_meta, test_meta)
    PRED[name] = (a.loc[dev_ids], b.loc[test_ids])
MAIN = ['phase_dropout_CNN', 'radiomics_noshape', 'cnn_plus_radiomics [phase_dropout_CNN]', 'full_fusion [phase_dropout_CNN]']
ALL_MODELS = MAIN + ['original_CNN', 'cnn_plus_radiomics [original_CNN]', 'full_fusion [original_CNN]']
y_dev, y_test = np.array([LABEL[p] for p in dev_ids]), np.array([LABEL[p] for p in test_ids])
def macro_auc(y, p):
    return float(roc_auc_score(y, p, multi_class='ovr', average='macro'))
CHECK = {m: round(macro_auc(y_test, PRED[m][1].to_numpy()), 3) for m in MAIN}
print('Held-out macro-AUC (must equal E2/E3: D-CNN 0.919, radiomics 0.902, CNN+radiomics D 0.952, full fusion D 0.951):', CHECK)
if IN_COLAB and (abs(CHECK['phase_dropout_CNN'] - 0.919) > 0.002 or abs(CHECK['radiomics_noshape'] - 0.902) > 0.002
                 or abs(CHECK['cnn_plus_radiomics [phase_dropout_CNN]'] - 0.952) > 0.002 or abs(CHECK['full_fusion [phase_dropout_CNN]'] - 0.951) > 0.002):
    raise RuntimeError('STOP — saved predictions do not reproduce E2/E3. Send this output.')

In [ ]:
# Cell 3 — E4a: truncation per phase and class; alignment of the independently centred phases inside the crops
def load_nii(path):                          # Week 2, unchanged
    arr = nib.load(str(path)).get_fdata().astype(np.float32)
    while arr.ndim > 3:
        if arr.shape[-1] == 1:
            arr = arr.squeeze(-1)
        elif arr.shape[-1] == 3:
            arr = arr.mean(axis=-1)
        else:
            arr = arr[..., 0]
    return arr

table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
table = table[table['patient_id'].isin(dev_ids + test_ids)].drop_duplicates(['patient_id', 'phase'])
def full_voxels(row):
    return row['patient_id'], row['phase'], int((load_nii(DATA_DIR / row['mask_path']) > 0).sum())
with ThreadPoolExecutor(8 if IN_COLAB else 2) as pool:
    FULL = {(p, ph): v for p, ph, v in pool.map(full_voxels, [r for _, r in table.iterrows()])}

def dice(a, b):
    s = a.sum() + b.sum()
    return float(2 * (a & b).sum() / s) if s else np.nan

trunc_rows = []
for pid in dev_ids + test_ids:
    for i, ph in enumerate(PHASE_ORDER):
        full, kept = FULL.get((pid, ph), np.nan), int(TUMOR_CROP[pid][i].sum())
        trunc_rows.append({'patient_id': pid, 'class': CLASSES[LABEL[pid]], 'phase': ph, 'full_voxels': full, 'crop_voxels': kept,
                           'retention': kept / full if full else np.nan, 'larger_than_crop': bool(full > 96 ** 3),
                           'tumour_dice_vs_C2': dice(TUMOR_CROP[pid][i], TUMOR_CROP[pid][2]) if ph != 'C2' else np.nan,
                           'liver_dice_vs_C2': dice(LIVER_CROP[pid][i], LIVER_CROP[pid][2]) if ph != 'C2' else np.nan})
TRUNC = pd.DataFrame(trunc_rows)
TRUNC.to_csv(OUT_DIR / 'E4a_truncation_per_patient_phase.csv', index=False)
agg = TRUNC.groupby(['class', 'phase']).agg(n=('patient_id', 'size'), median_retention=('retention', 'median'),
                                             patients_below_0_9=('retention', lambda s: int((s < .9).sum())),
                                             larger_than_crop=('larger_than_crop', 'sum'),
                                             tumour_dice_vs_C2=('tumour_dice_vs_C2', 'median'), liver_dice_vs_C2=('liver_dice_vs_C2', 'median'))
vw = TRUNC.groupby(['class', 'phase']).apply(lambda g: g['crop_voxels'].sum() / g['full_voxels'].sum())
agg['volume_weighted_retention'] = vw
E4A = agg.reset_index()
E4A.to_csv(OUT_DIR / 'E4a_truncation_summary.csv', index=False)
overall = {'volume_weighted_retention_all': float(TRUNC['crop_voxels'].sum() / TRUNC['full_voxels'].sum()),
           'tumours_larger_than_crop_C2': TRUNC[(TRUNC['phase'] == 'C2') & TRUNC['larger_than_crop']].groupby('class').size().to_dict()}
overall['patient_phases_with_retention_above_1'] = int((TRUNC['retention'] > 1.001).sum())   # should be 0 (crop is part of the full mask)
print('E4a overall:', overall)
print(E4A.round(3).to_string(index=False))

In [ ]:
# Cell 4 — E4b: internal results by tumour-size thirds (cut-offs from the 222 development patients) and truncation
def exact_ci(k, n):
    low = 0.0 if k == 0 else float(beta.ppf(0.025, k, n - k + 1))
    high = 1.0 if k == n else float(beta.ppf(0.975, k + 1, n - k))
    return low, high

CUTS = np.percentile([SIZE[p] for p in dev_ids], [100 / 3, 200 / 3])
def third(pid):
    return 'small' if SIZE[pid] <= CUTS[0] else ('medium' if SIZE[pid] <= CUTS[1] else 'large')
retention_c2 = TRUNC[TRUNC['phase'] == 'C2'].set_index('patient_id')['retention']
def stratum_rows(ids, probs, model, split_name, grouping, groups):
    rows = []
    for g in sorted(set(groups.values())):
        members = [p for p in ids if groups[p] == g]
        y = np.array([LABEL[p] for p in members])
        p = probs.loc[members].to_numpy()
        pred = p.argmax(axis=1)
        counts = np.bincount(y, minlength=3)
        row = {'model': model, 'split': split_name, 'grouping': grouping, 'group': g, 'n': len(members),
               'n_HCC': int(counts[0]), 'n_ICC': int(counts[1]), 'n_CHCC': int(counts[2]),
               'macro_auc': macro_auc(y, p) if counts.min() >= 3 else np.nan, 'accuracy': float((pred == y).mean())}
        for c, name in enumerate(CLASSES):
            k, n = int(((pred == c) & (y == c)).sum()), int((y == c).sum())
            row[f'recall_{name}'] = k / n if n else np.nan
        rows.append(row)
    return rows
size_groups = {p: third(p) for p in dev_ids + test_ids}
trunc_groups = {p: ('truncated' if retention_c2.get(p, 1.0) < 0.9 else 'not_truncated') for p in dev_ids + test_ids}
E4B = []
for model in ALL_MODELS:
    for split_name, ids, idx in [('oof_222', dev_ids, 0), ('test_56', test_ids, 1)]:
        E4B += stratum_rows(ids, PRED[model][idx], model, split_name, 'size_third', size_groups)
        E4B += stratum_rows(ids, PRED[model][idx], model, split_name, 'truncation_C2', trunc_groups)
E4B = pd.DataFrame(E4B)
E4B.to_csv(OUT_DIR / 'E4b_internal_by_size_and_truncation.csv', index=False)
print('Size cut-offs (development thirds, voxels):', CUTS.round(0).tolist())
print('Class mix per size third (development):', pd.crosstab(pd.Series(size_groups).loc[dev_ids], pd.Series({p: CLASSES[LABEL[p]] for p in dev_ids})).to_dict())
show = E4B[(E4B['split'] == 'oof_222') & E4B['model'].isin(MAIN)]
print(show[['model', 'grouping', 'group', 'n', 'macro_auc', 'accuracy', 'recall_HCC', 'recall_ICC', 'recall_CHCC']].round(3).to_string(index=False))

In [ ]:
# Cell 5 — E4c: MCT-LTDiag (all 203) by crop retention and by size thirds (HCC vs ICC AUC), descriptive
P203 = pd.read_csv(INPUTS['e8a_predictions_203'][0], dtype={'patient_id': str})
P203 = P203[P203['variant'] == 'primary']
geo = pd.concat([pd.read_csv(INPUTS['e6_crop_geometry'][0], dtype={'patient_id': str}),
                 pd.read_csv(INPUTS['e8a_crop_geometry_2'][0], dtype={'patient_id': str})])
geo = geo[geo['variant'] == 'primary'].drop_duplicates('patient_id').set_index('patient_id')
facts = pd.concat([pd.read_csv(INPUTS['e6_facts'][0], dtype={'patient_id': str}), pd.read_csv(INPUTS['e8a_facts_2'][0], dtype={'patient_id': str})])
facts = facts[facts['tumor_volume_ml'].notna()].drop_duplicates('patient_id', keep='last').set_index('patient_id')
MCT_IDS = sorted(P203['patient_id'].unique())
TRUE_MCT = P203.drop_duplicates('patient_id').set_index('patient_id').loc[MCT_IDS, 'true_class']
Y_MCT = (TRUE_MCT == 'HCC').astype(int).to_numpy()
vol = facts.loc[MCT_IDS, 'tumor_volume_ml']
mct_cuts = np.percentile(vol, [100 / 3, 200 / 3])
MCT_GROUPS = {'retention': {p: ('retention_below_0.9' if geo.loc[p, 'tumor_in_crop'] < .9 else 'retention_0.9_or_more') for p in MCT_IDS},
              'size_third_MCT': {p: ('small' if vol[p] <= mct_cuts[0] else 'medium' if vol[p] <= mct_cuts[1] else 'large') for p in MCT_IDS}}
def hcc_score(model):
    t = P203[P203['model'] == model].set_index('patient_id').loc[MCT_IDS]
    return (t['p_HCC'] / (t['p_HCC'] + t['p_ICC']).clip(lower=1e-12)).to_numpy()
E4C = []
for model in ALL_MODELS:
    s = hcc_score(model)
    for grouping, groups in MCT_GROUPS.items():
        for g in sorted(set(groups.values())):
            k = np.array([groups[p] == g for p in MCT_IDS])
            y = Y_MCT[k]
            E4C.append({'model': model, 'grouping': grouping, 'group': g, 'n': int(k.sum()), 'n_HCC': int(y.sum()), 'n_ICC': int((1 - y).sum()),
                        'auc_hcc_vs_icc': float(roc_auc_score(y, s[k])) if 0 < y.sum() < len(y) else np.nan})
E4C = pd.DataFrame(E4C)
E4C.to_csv(OUT_DIR / 'E4c_mct203_by_retention_and_size.csv', index=False)
print('MCT size cut-offs (ml):', mct_cuts.round(1).tolist())
print(E4C[E4C['model'].isin(MAIN)].round(3).to_string(index=False))

In [ ]:
# Cell 6 — E7b: final internal statistics (held-out 56 and out-of-fold 222); DeLong with Benjamini-Hochberg
def compute_midrank(values):
    order = np.argsort(values)
    sorted_values = values[order]
    n_values = len(values)
    midranks = np.zeros(n_values, dtype=np.float64)
    index = 0
    while index < n_values:
        end = index
        while end < n_values and sorted_values[end] == sorted_values[index]:
            end += 1
        midranks[index:end] = 0.5 * (index + end - 1) + 1
        index = end
    output = np.empty(n_values, dtype=np.float64)
    output[order] = midranks
    return output

def delong_test(ground_truth, predictions_one, predictions_two):          # E2, unchanged
    ordering = np.argsort(-ground_truth)
    predictions = np.vstack([predictions_one, predictions_two])[:, ordering]
    positives = int(np.sum(ground_truth == 1))
    negatives = len(ground_truth) - positives
    tx = np.empty((2, positives)); ty = np.empty((2, negatives)); tz = np.empty((2, positives + negatives))
    for model_index in range(2):
        tx[model_index] = compute_midrank(predictions[model_index, :positives])
        ty[model_index] = compute_midrank(predictions[model_index, positives:])
        tz[model_index] = compute_midrank(predictions[model_index])
    aucs = tz[:, :positives].sum(axis=1) / positives / negatives - (positives + 1.0) / 2.0 / negatives
    v01 = (tz[:, :positives] - tx) / negatives
    v10 = 1.0 - (tz[:, positives:] - ty) / positives
    covariance = np.cov(v01) / positives + np.cov(v10) / negatives
    contrast = np.array([1.0, -1.0])
    variance = float(contrast @ covariance @ contrast)
    difference = float(aucs[0] - aucs[1])
    if variance <= 0:
        return {'auc_one': float(aucs[0]), 'auc_two': float(aucs[1]), 'difference': difference, 'p': 1.0}
    return {'auc_one': float(aucs[0]), 'auc_two': float(aucs[1]), 'difference': difference,
            'p': float(2.0 * norm.sf(abs(difference / np.sqrt(variance))))}

def adjust_pvalues(raw_pvalues):                                          # E2, unchanged
    raw = np.asarray(raw_pvalues, dtype=float)
    n_tests = len(raw)
    bonferroni = np.minimum(raw * n_tests, 1.0)
    order = np.argsort(raw)
    adjusted = raw[order] * n_tests / np.arange(1, n_tests + 1)
    adjusted = np.minimum(np.minimum.accumulate(adjusted[::-1])[::-1], 1.0)
    benjamini_hochberg = np.empty(n_tests)
    benjamini_hochberg[order] = adjusted
    return bonferroni, benjamini_hochberg

rng = np.random.default_rng(SEED)
BOOT56 = [rng.integers(0, len(test_ids), len(test_ids)) for _ in range(N_BOOT)]
def boot_ci(y, p):
    vals = [macro_auc(y[b], p[b]) for b in BOOT56 if len(np.unique(y[b])) == 3]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
E7B = []
for model in ALL_MODELS:
    p_test, p_oof = PRED[model][1].to_numpy(), PRED[model][0].to_numpy()
    pred = p_test.argmax(axis=1)
    fold_auc = [macro_auc(np.array([LABEL[x] for x in fs['val_ids'] if x in dev_ids]),
                          PRED[model][0].loc[[x for x in fs['val_ids'] if x in dev_ids]].to_numpy()) for fs in fold_splits]
    low, high = boot_ci(y_test, p_test)
    row = {'model': model, 'test_macro_auc': macro_auc(y_test, p_test), 'boot_low': low, 'boot_high': high,
           'test_accuracy': float(accuracy_score(y_test, pred)), 'test_macro_f1': float(f1_score(y_test, pred, average='macro')),
           'oof_macro_auc': macro_auc(y_dev, p_oof), 'cv_fold_auc_mean': float(np.mean(fold_auc)), 'cv_fold_auc_sd': float(np.std(fold_auc))}
    for c, name in enumerate(CLASSES):
        tp, fn = int(((pred == c) & (y_test == c)).sum()), int(((pred != c) & (y_test == c)).sum())
        tn, fp = int(((pred != c) & (y_test != c)).sum()), int(((pred == c) & (y_test != c)).sum())
        row[f'sens_{name}'], (row[f'sens_{name}_low'], row[f'sens_{name}_high']) = tp / (tp + fn), exact_ci(tp, tp + fn)
        row[f'spec_{name}'], (row[f'spec_{name}_low'], row[f'spec_{name}_high']) = tn / (tn + fp), exact_ci(tn, tn + fp)
    E7B.append(row)
E7B = pd.DataFrame(E7B)
E7B.to_csv(OUT_DIR / 'E7b_internal_final_metrics.csv', index=False)

pairs = [(MAIN[a], MAIN[b]) for a in range(len(MAIN)) for b in range(a + 1, len(MAIN))]
tests = []
for a, b in pairs:
    for c, name in enumerate(CLASSES):
        r = delong_test((y_test == c).astype(int), PRED[a][1].to_numpy()[:, c], PRED[b][1].to_numpy()[:, c])
        tests.append({'model_a': a, 'model_b': b, 'class': name, **r})
DELONG = pd.DataFrame(tests)
DELONG['p_bonferroni'], DELONG['p_bh'] = adjust_pvalues(DELONG['p'])
DELONG.to_csv(OUT_DIR / 'E7b_delong_main_pairs_bh.csv', index=False)
paired = []
for a, b in pairs:
    pa, pb = PRED[a][1].to_numpy(), PRED[b][1].to_numpy()
    d = [macro_auc(y_test[i], pa[i]) - macro_auc(y_test[i], pb[i]) for i in BOOT56 if len(np.unique(y_test[i])) == 3]
    paired.append({'model_a': a, 'model_b': b, 'difference': macro_auc(y_test, pa) - macro_auc(y_test, pb),
                   'boot_low': float(np.percentile(d, 2.5)), 'boot_high': float(np.percentile(d, 97.5)),
                   'boot_two_sided_p': float(min(1.0, 2 * min(np.mean(np.array(d) <= 0), np.mean(np.array(d) >= 0))))})
PAIRED_INT = pd.DataFrame(paired)
PAIRED_INT.to_csv(OUT_DIR / 'E7b_paired_bootstrap_main_pairs.csv', index=False)
print(E7B[['model', 'test_macro_auc', 'boot_low', 'boot_high', 'cv_fold_auc_mean', 'cv_fold_auc_sd', 'test_accuracy']].round(3).to_string(index=False))
print(f'DeLong main pairs x classes: {len(DELONG)} tests; significant after BH (p < 0.05): {int((DELONG["p_bh"] < .05).sum())}')

In [ ]:
# Cell 7 — E7c: calibration (Brier, intercept, slope, reliability), internal held-out and MCT-LTDiag (HCC vs ICC)
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def logistic_fit(x, y, offset=None, slope=True, iters=100):
    """Unpenalised logistic regression y ~ a (+ b*x) (+ offset), by Newton steps. Returns (a, b)."""
    a, b = 0.0, 1.0 if not slope else 0.0
    off = np.zeros_like(x) if offset is None else offset
    for _ in range(iters):
        eta = a + (b * x if slope else 0) + off
        p = 1 / (1 + np.exp(-eta))
        w = p * (1 - p) + 1e-12
        if slope:
            X = np.column_stack([np.ones_like(x), x])
            H = X.T @ (X * w[:, None]); g = X.T @ (y - p)
            step = np.linalg.solve(H + 1e-9 * np.eye(2), g)
            a, b = a + step[0], b + step[1]
        else:
            step = (y - p).sum() / w.sum()
            a += step
        if np.max(np.abs(step)) < 1e-10:
            break
    return a, b

def calibration(y, p):
    intercept, _ = logistic_fit(logit(p), y, offset=logit(p), slope=False)      # calibration-in-the-large (slope fixed at 1)
    a, slope = logistic_fit(logit(p), y, slope=True)
    bins = np.clip((p * 5).astype(int), 0, 4)
    table = [{'bin': f'{k / 5:.1f}-{(k + 1) / 5:.1f}', 'n': int((bins == k).sum()),
              'mean_predicted': float(p[bins == k].mean()) if (bins == k).any() else np.nan,
              'observed': float(y[bins == k].mean()) if (bins == k).any() else np.nan} for k in range(5)]
    return {'brier': float(np.mean((p - y) ** 2)), 'intercept': float(intercept), 'slope': float(slope)}, table

CAL, REL = [], []
for model in ALL_MODELS:
    p = PRED[model][1].to_numpy()
    multi_brier = float(np.mean(((p - np.eye(3)[y_test]) ** 2).sum(axis=1)))
    for c, name in enumerate(CLASSES):
        stats, table = calibration((y_test == c).astype(float), p[:, c])
        CAL.append({'model': model, 'cohort': 'internal_test_56', 'target': f'{name}_vs_rest', 'multiclass_brier': multi_brier, **stats})
        REL += [{'model': model, 'cohort': 'internal_test_56', 'target': f'{name}_vs_rest', **t} for t in table]
    s = hcc_score(model)
    stats, table = calibration(Y_MCT.astype(float), s)
    CAL.append({'model': model, 'cohort': 'MCT_LTDiag_203', 'target': 'HCC_vs_ICC (renormalised)', 'multiclass_brier': np.nan, **stats})
    REL += [{'model': model, 'cohort': 'MCT_LTDiag_203', 'target': 'HCC_vs_ICC (renormalised)', **t} for t in table]
CAL, REL = pd.DataFrame(CAL), pd.DataFrame(REL)
CAL.to_csv(OUT_DIR / 'E7c_calibration.csv', index=False)
REL.to_csv(OUT_DIR / 'E7c_reliability_tables.csv', index=False)
print(CAL[CAL['model'].isin(MAIN)].round(3).to_string(index=False))

In [ ]:
# Cell 8 — E7d: optimism of the fusion meta-learner (apparent vs CV vs held-out; Harrell bootstrap)
def fit_single(columns, frame):
    X = prepare(frame[columns].to_numpy())
    imp = SimpleImputer(strategy='median').fit(X)
    sc = StandardScaler().fit(imp.transform(X))
    m = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED).fit(sc.transform(imp.transform(X)), frame['label'].astype(int))
    return lambda f: m.predict_proba(sc.transform(imp.transform(prepare(f[columns].to_numpy()))))
E7D = []
rng_o = np.random.default_rng(SEED)
for name, cols in {'cnn_plus_radiomics [phase_dropout_CNN]': W4 + W5, 'full_fusion [phase_dropout_CNN]': W4 + W5 + ['age', 'gender']}.items():
    dev = oof_d.set_index('patient_id').loc[dev_ids].reset_index()
    yd = dev['label'].astype(int).to_numpy()
    apparent = macro_auc(yd, fit_single(cols, dev)(dev))
    optimism = []
    for _ in range(N_OPTIMISM):
        idx = rng_o.integers(0, len(dev), len(dev))
        boot = dev.iloc[idx].reset_index(drop=True)
        if len(np.unique(boot['label'])) < 3:
            continue
        model = fit_single(cols, boot)
        optimism.append(macro_auc(boot['label'].astype(int).to_numpy(), model(boot)) - macro_auc(yd, model(dev)))
    E7D.append({'model': name, 'apparent_auc': apparent, 'harrell_optimism': float(np.mean(optimism)),
                'optimism_corrected_auc': apparent - float(np.mean(optimism)),
                'cv_fold_auc_mean': float(E7B.set_index('model').loc[name, 'cv_fold_auc_mean']),
                'held_out_auc': float(E7B.set_index('model').loc[name, 'test_macro_auc']), 'bootstraps': len(optimism)})
E7D = pd.DataFrame(E7D)
E7D.to_csv(OUT_DIR / 'E7d_optimism.csv', index=False)
print(E7D.round(4).to_string(index=False))
print('Note: optimism here concerns the meta-learner only; its inputs are already out-of-fold predictions.')

In [ ]:
# Cell 9 — Save summary and the block to send back
summary = {'run_id': RUN_ID, 'notebook': 'E47_size_truncation_calibration_statistics.ipynb', 'versions': VERSIONS,
           'input_sha256': input_hashes, 'e8a_run_used': E8A_RUN.name, 'reproduction_check_test_auc': CHECK,
           'e4a_overall': overall, 'size_cutoffs_dev_voxels': CUTS.tolist(), 'mct_size_cutoffs_ml': mct_cuts.tolist(),
           'delong_tests': len(DELONG), 'delong_significant_bh': int((DELONG['p_bh'] < .05).sum()),
           'protocol': '00 log sections 19-20 (approved 17 Oct 2026); descriptive; no training or tuning'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest_out = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)}
                for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest_out).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E47_{RUN_ID} | E8a run used: {E8A_RUN.name} | held-out AUC check: {CHECK}')
print('E4a:', overall)
print(E4A[E4A['phase'] == 'C2'][['class', 'n', 'median_retention', 'volume_weighted_retention', 'patients_below_0_9', 'larger_than_crop']].round(3).to_string(index=False))
print('Interphase alignment inside crops (median Dice vs C2):', E4A.groupby('phase')[['tumour_dice_vs_C2', 'liver_dice_vs_C2']].median().round(3).to_dict())
print('\nE4b (out-of-fold 222), main models:')
print(E4B[(E4B['split'] == 'oof_222') & E4B['model'].isin(MAIN)][['model', 'grouping', 'group', 'n', 'macro_auc', 'recall_HCC', 'recall_ICC', 'recall_CHCC']].round(3).to_string(index=False))
print('\nE4c (MCT-LTDiag 203), main models:')
print(E4C[E4C['model'].isin(MAIN)].round(3).to_string(index=False))
print('\nE7b held-out 56:')
print(E7B[['model', 'test_macro_auc', 'boot_low', 'boot_high', 'cv_fold_auc_mean', 'cv_fold_auc_sd']].round(3).to_string(index=False))
print(f'DeLong (main pairs x classes): {len(DELONG)} tests, significant after BH: {int((DELONG["p_bh"] < .05).sum())}')
print(PAIRED_INT.round(3).to_string(index=False))
print('\nE7c calibration (main models):')
print(CAL[CAL['model'].isin(MAIN)][['model', 'cohort', 'target', 'brier', 'intercept', 'slope']].round(3).to_string(index=False))
print('\nE7d optimism:')
print(E7D.round(4).to_string(index=False))
print('==============================================================')